In [1]:
!nvidia-smi

Sun May 10 14:36:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**Problem 1**       
Parallel DNA Complement Generation

In [2]:
%%writefile dna_complement.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

__global__ void complement_kernel(const char* input, char* output, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        char base = input[idx];
        if (base == 'A') output[idx] = 'T';
        else if (base == 'T') output[idx] = 'A';
        else if (base == 'C') output[idx] = 'G';
        else if (base == 'G') output[idx] = 'C';
    }
}

void run_experiment(int n, char* h_input, int block_size) {
    char *d_input, *d_output;
    char *h_output = (char*)malloc(n * sizeof(char));

    cudaMalloc((void**)&d_input, n * sizeof(char));
    cudaMalloc((void**)&d_output, n * sizeof(char));

    cudaMemcpy(d_input, h_input, n * sizeof(char), cudaMemcpyHostToDevice);

    int grid_size = (n + block_size - 1) / block_size;
    int total_threads = grid_size * block_size;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    complement_kernel<<<grid_size, block_size>>>(d_input, d_output, n);
    cudaEventRecord(stop);

    cudaEventSynchronize(stop);
    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(h_output, d_output, n * sizeof(char), cudaMemcpyDeviceToHost);

    printf("--- Experiment: Block Size %d ---\n", block_size);
    printf("Total Threads Launched: %d\n", total_threads);
    printf("Kernel Execution Time: %f ms\n", ms);

    printf("Original (first 50): ");
    for(int i = 0; i < 50; i++) printf("%c", h_input[i]);
    printf("\nComplement (first 50): ");
    for(int i = 0; i < 50; i++) printf("%c", h_output[i]);
    printf("\n\n");

    cudaFree(d_input);
    cudaFree(d_output);
    free(h_output);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
}

int main() {
    int n = 10000000;
    char* h_input = (char*)malloc(n * sizeof(char));
    char bases[] = {'A', 'C', 'G', 'T'};

    srand(time(NULL));
    for (int i = 0; i < n; i++) {
        h_input[i] = bases[rand() % 4];
    }

    int block_sizes[] = {64, 128, 256, 512};
    for (int i = 0; i < 4; i++) {
        run_experiment(n, h_input, block_sizes[i]);
    }

    free(h_input);
    return 0;
}

Writing dna_complement.cu


In [3]:
!nvcc -Wno-deprecated-gpu-targets -o dna_complement dna_complement.cu
!./dna_complement

--- Experiment: Block Size 64 ---
Total Threads Launched: 10000000
Kernel Execution Time: 112.523071 ms
Original (first 50): GTGGAAACTACACGACCACAAGAGATCTAATGTCCTCCAACCATAAACAC
Complement (first 50): CACCTTTGATGTGCTGGTGTTCTCTAGATTACAGGAGGTTGGTATTTGTG

--- Experiment: Block Size 128 ---
Total Threads Launched: 10000000
Kernel Execution Time: 0.274464 ms
Original (first 50): GTGGAAACTACACGACCACAAGAGATCTAATGTCCTCCAACCATAAACAC
Complement (first 50): CACCTTTGATGTGCTGGTGTTCTCTAGATTACAGGAGGTTGGTATTTGTG

--- Experiment: Block Size 256 ---
Total Threads Launched: 10000128
Kernel Execution Time: 0.269856 ms
Original (first 50): GTGGAAACTACACGACCACAAGAGATCTAATGTCCTCCAACCATAAACAC
Complement (first 50): CACCTTTGATGTGCTGGTGTTCTCTAGATTACAGGAGGTTGGTATTTGTG

--- Experiment: Block Size 512 ---
Total Threads Launched: 10000384
Kernel Execution Time: 0.275104 ms
Original (first 50): GTGGAAACTACACGACCACAAGAGATCTAATGTCCTCCAACCATAAACAC
Complement (first 50): CACCTTTGATGTGCTGGTGTTCTCTAGATTACAGGAGGTTGGTATTTGTG


**Problem 2**                                       
Parallel DNA Base Frequency Analysis


In [4]:
%%writefile nucleotide_frequency.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

__global__ void count_kernel(const char* input, int n, unsigned int* d_counts) {
    int gid = blockIdx.x * blockDim.x + threadIdx.x;
    if (gid < n) {
        char base = input[gid];
        if (base == 'A')      atomicAdd(&d_counts[0], 1);
        else if (base == 'C') atomicAdd(&d_counts[1], 1);
        else if (base == 'G') atomicAdd(&d_counts[2], 1);
        else if (base == 'T') atomicAdd(&d_counts[3], 1);
    }
}

void run_frequency_experiment(int n, char* h_input, int block_size) {
    unsigned int *d_counts;
    char *d_input;
    unsigned int h_counts[4] = {0, 0, 0, 0};

    cudaMalloc((void**)&d_input, n * sizeof(char));
    cudaMalloc((void**)&d_counts, 4 * sizeof(unsigned int));

    cudaMemcpy(d_input, h_input, n * sizeof(char), cudaMemcpyHostToDevice);
    cudaMemcpy(d_counts, h_counts, 4 * sizeof(unsigned int), cudaMemcpyHostToDevice);

    int grid_size = (n + block_size - 1) / block_size;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    count_kernel<<<grid_size, block_size>>>(d_input, n, d_counts);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(h_counts, d_counts, 4 * sizeof(unsigned int), cudaMemcpyDeviceToHost);

    unsigned int total = h_counts[0] + h_counts[1] + h_counts[2] + h_counts[3];

    printf("Block Size: %d | Blocks: %d\n", block_size, grid_size);
    printf("Counts: A:%u C:%u G:%u T:%u\n", h_counts[0], h_counts[1], h_counts[2], h_counts[3]);
    printf("Total Sum: %u | Time: %f ms\n\n", total, ms);

    cudaFree(d_input);
    cudaFree(d_counts);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
}

int main() {
    int n = 20000000;
    char* h_input = (char*)malloc(n * sizeof(char));
    char bases[] = {'A', 'C', 'G', 'T'};

    srand(time(NULL));
    for (int i = 0; i < n; i++) h_input[i] = bases[rand() % 4];

    int block_sizes[] = {64, 128, 256, 512};
    for (int i = 0; i < 4; i++) run_frequency_experiment(n, h_input, block_sizes[i]);

    free(h_input);
    return 0;
}

Writing nucleotide_frequency.cu


In [5]:
!nvcc -o nucleotide_frequency nucleotide_frequency.cu
!./nucleotide_frequency

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Block Size: 64 | Blocks: 312500
Counts: A:5001907 C:5000014 G:4997645 T:5000434
Total Sum: 20000000 | Time: 25.916416 ms

Block Size: 128 | Blocks: 156250
Counts: A:5001907 C:5000014 G:4997645 T:5000434
Total Sum: 20000000 | Time: 3.684288 ms

Block Size: 256 | Blocks: 78125
Counts: A:5001907 C:5000014 G:4997645 T:5000434
Total Sum: 20000000 | Time: 3.683936 ms

Block Size: 512 | Blocks: 39063
Counts: A:5001907 C:5000014 G:4997645 T:5000434
Total Sum: 20000000 | Time: 3.681920 ms



**Problem 3**                   
Codon and Warp Mapping Analysis


In [6]:
%%writefile codon_mapping.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void mapping_kernel(int n) {
    int gid = blockIdx.x * blockDim.x + threadIdx.x;

    if (gid < n) {
        // Calculations as per task requirements
        int codon_id = gid / 3;
        int warp_id = threadIdx.x / 32;
        int lane_id = threadIdx.x % 32;

        // Print only first few threads to avoid flooding the console
        if (gid < 10) {
            printf("Thread %d: Codon %d, Warp %d, Lane %d\n", gid, codon_id, warp_id, lane_id);
        }
    }
}

int main() {
    int n = 24000000;
    int block_size = 256;
    int grid_size = (n + block_size - 1) / block_size;

    printf("Total DNA Length: %d\n", n);
    printf("Total Codons: %d\n", n / 3);
    printf("Total Threads: %d\n", n);
    printf("Total Warps: %d\n", (n + 31) / 32);
    printf("\nMapping samples:\n");

    mapping_kernel<<<grid_size, block_size>>>(n);
    cudaDeviceSynchronize();

    return 0;
}

Writing codon_mapping.cu


In [7]:
!nvcc -o codon_mapping codon_mapping.cu
!./codon_mapping

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Total DNA Length: 24000000
Total Codons: 8000000
Total Threads: 24000000
Total Warps: 750000

Mapping samples:
Thread 0: Codon 0, Warp 0, Lane 0
Thread 1: Codon 0, Warp 0, Lane 1
Thread 2: Codon 0, Warp 0, Lane 2
Thread 3: Codon 1, Warp 0, Lane 3
Thread 4: Codon 1, Warp 0, Lane 4
Thread 5: Codon 1, Warp 0, Lane 5
Thread 6: Codon 2, Warp 0, Lane 6
Thread 7: Codon 2, Warp 0, Lane 7
Thread 8: Codon 2, Warp 0, Lane 8
Thread 9: Codon 3, Warp 0, Lane 9
